# 🪪 Entrenar: Detector de Placas Vehiculares (YOLOv8)
## Para usar con nuestro Plate Reader API

Este notebook entrena **YOLOv8** para detectar la ubicación de placas en imágenes.
Después usamos PaddleOCR para leer el texto.

**⚠️ Antes de empezar:**
- Entorno de ejecución → Cambiar tipo → **GPU** ✅
- Ejecuta las celdas **UNA POR UNA**
- Tiempo estimado: ~2-3 horas

---

In [ ]:
# CELDA 1: Montar Drive + instalar
from google.colab import drive
drive.mount('/content/drive')
!pip install ultralytics kagglehub -q

In [ ]:
# CELDA 2: Descargar dataset de placas (CCRL)
import kagglehub

print("Descargando dataset CCRL (Chinese City Parking)...")
path = kagglehub.dataset_download("city-foliage/ccrl-chinese-city-parking-dataset")
print(f"Dataset en: {path}")

In [ ]:
# CELDA 3: Preparar dataset en formato YOLOv8
import os, shutil, random
from pathlib import Path
from tqdm import tqdm

data_path = Path(path)
out = Path('/content/plates_yolo')

# Crear estructura YOLO
for split in ['train', 'val']:
    (out / 'images' / split).mkdir(parents=True, exist_ok=True)
    (out / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Buscar imágenes y anotaciones
images = list(data_path.rglob('*.jpg')) + list(data_path.rglob('*.png'))
random.shuffle(images)

# 80% train, 20% val
split_idx = int(len(images) * 0.8)
train_imgs = images[:split_idx]
val_imgs = images[split_idx:]

def process_images(img_list, split_name):
    for img_path in tqdm(img_list):
        # Copiar imagen
        dst_img = out / 'images' / split_name / img_path.name
        shutil.copy2(img_path, dst_img)
        
        # Buscar label correspondiente
        label_path = img_path.with_suffix('.txt')
        if label_path.exists():
            shutil.copy2(label_path, out / 'labels' / split_name / label_path.name)

process_images(train_imgs, 'train')
process_images(val_imgs, 'val')

print(f"✅ Train: {len(train_imgs)} | Val: {len(val_imgs)}")
print(f"   Labels: {len(list((out/'labels'/'train').iterdir()))} archivos")

In [ ]:
# CELDA 4: Crear archivo dataset.yaml
yaml_content = f"""
train: {out}/images/train
val: {out}/images/val
nc: 1
names: ['license_plate']
"""

with open(out / 'dataset.yaml', 'w') as f:
    f.write(yaml_content)
print("✅ dataset.yaml creado")

In [ ]:
# CELDA 5: ENTRENAR YOLOv8 (detección, ~2-3h con GPU)
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # modelo nano (~6MB)

results = model.train(
    data=str(out / 'dataset.yaml'),
    epochs=100,
    imgsz=640,
    batch=16,
    device='cuda',
    workers=4,
    lr0=0.01,
    patience=15,
    name='plate_detector',
    project='/content/drive/MyDrive/vehicle_ai',
    exist_ok=True
)
print("✅ Entrenamiento completado")

In [ ]:
# CELDA 6: Exportar y subir al VPS
model.export(format='onnx', imgsz=640)

print("✅ Modelo exportado")
print("📁 Ubicación:")
print("   /content/drive/MyDrive/vehicle_ai/plate_detector/weights/best.onnx")
print()
print("📥 Para subir al VPS:")
print("   1. Descarga best.onnx de tu Drive")
print("   2. Súbelo al VPS:")
print("   scp best.onnx root@2.25.128.81:/opt/docker-projects/")
print("   plate_recognition/plate_reader/models/plate_detector.pt")